In [4]:
# import libraries
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None) # display all columns in the dataframe

In [5]:
patients = pd.read_csv('../data/processed/patients_cleaned.csv', parse_dates=['registration_date'])
vitals = pd.read_csv('../data/processed/vital_signs_cleaned.csv', parse_dates=['timestamp'])
history = pd.read_csv('../data/processed/clinical_history_cleaned.csv')
labs = pd.read_csv('../data/processed/laboratory_results_cleaned.csv', parse_dates=['timestamp'])
outcomes = pd.read_csv('../data/processed/sepsis_outcomes_cleaned.csv', parse_dates=['diagnosis_time'])

tables = {'patients': patients, 'vitals': vitals, 'history': history, 'labs': labs, 'outcomes': outcomes}
for name, df in tables.items():
    print(f"{name} shape: {df.shape}")

patients shape: (5000, 5)
vitals shape: (99737, 8)
history shape: (12459, 6)
labs shape: (20034, 8)
outcomes shape: (5000, 6)


In [6]:
# Sort vitals and labs dataframes by patient_id and timestamp
vitals = vitals.sort_values(['patient_id', 'timestamp']).reset_index(drop=True)
labs = labs.sort_values(['patient_id', 'timestamp']).reset_index(drop=True)

In [7]:
vitals.columns

Index(['observation_id', 'patient_id', 'timestamp', 'heart_rate',
       'temperature', 'oxygen_saturation', 'respiratory_rate',
       'blood_pressure'],
      dtype='str')

In [8]:
labs.columns

Index(['lab_id', 'patient_id', 'timestamp', 'white_cell_count', 'crp',
       'lactate', 'creatinine', 'platelet_count'],
      dtype='str')

In [9]:
vital_cols = [
  'heart_rate',
  'temperature',
  'oxygen_saturation',
  'respiratory_rate',
  'blood_pressure'
]

lab_cols = [
  'white_cell_count',
  'crp',
  'lactate',
  'creatinine',
  'platelet_count',
]

In [10]:
outcomes.columns

Index(['outcome_id', 'patient_id', 'sepsis_event', 'diagnosis_time',
       'hospitalisation_required', 'outcome_status'],
      dtype='str')

In [11]:
# Feature engineering for prediction time
rng = np.random.default_rng(7) # set random seed for reproducibility

def get_prediction_time(row, vitals_df):
  if row['sepsis_event']:
    return row['diagnosis_time'] - pd.Timedelta(hours=9)
  pv = vitals_df[vitals_df['patient_id'] == row['patient_id']]
  start, end = pv['timestamp'].min(), pv['timestamp'].max()
  span_hours = max((end - start).total_seconds() / 3600, 1)
  offset_hours = rng.uniform(0.4, 0.9) * span_hours
  return start + pd.Timedelta(hours=offset_hours)

outcomes = outcomes.copy()
outcomes['prediction_time'] = outcomes.apply(get_prediction_time, vitals_df=vitals, axis=1)
outcomes[['patient_id', 'sepsis_event', 'diagnosis_time', 'prediction_time']].head(15)

,patient_id,sepsis_event,diagnosis_time,prediction_time
0,1,True,2024-12-13 05:36:00,2024-12-12 20:36:00.000000000
1,2,False,NaT,2025-08-28 06:26:55.786260611
2,3,True,2024-07-19 13:24:00,2024-07-19 04:24:00.000000000
3,4,True,2024-08-21 21:07:00,2024-08-21 12:07:00.000000000
4,5,False,NaT,2024-04-11 14:53:44.568198471
5,6,False,NaT,2024-06-10 07:48:58.256127454
6,7,True,2024-01-29 10:48:00,2024-01-29 01:48:00.000000000
7,8,False,NaT,2025-08-15 03:04:57.752682750
8,9,False,NaT,2025-09-11 12:46:43.210235189
9,10,True,2024-11-02 06:47:00,2024-11-01 21:47:00.000000000


In [12]:
# Feature engineering for vitals and labs
LOOKBACK_HOURS = 6

def vitals_features(pid, cutoff, df):
  window=df[
    (df['patient_id'] == pid) & (df['timestamp'] <= cutoff) & (df['timestamp'] >= cutoff - pd.Timedelta(hours=LOOKBACK_HOURS))
  ]

  if window.empty:
    window = df[(df['patient_id'] == pid) & (df['timestamp'] <= cutoff)].tail(1)
  feats={}
  for col in vital_cols:
    vals = window[col]
    feats[f'{col}_mean'] = vals.mean()
    feats[f'{col}_min'] = vals.min()
    feats[f'{col}_max'] = vals.max()
    feats[f'{col}_std'] = vals.std() if len(vals) > 1 else 0.0
    feats[f'{col}_last'] = vals.iloc[-1] if len(vals) else np.nan

    if len(window) > 1:
      hours = (window['timestamp'].iloc[-1] - window['timestamp'].iloc[0]).total_seconds() / 3600
      feats[f'{col}_rate_per_hour'] = (vals.iloc[-1] - vals.iloc[0]) / hours if hours > 0 else 0.0
    else:
      feats[f'{col}_rate_per_hour'] = 0.0
      
  return feats

vitals_features_rows = [
  {'patient_id': pid, **vitals_features(pid, cutoff, vitals)}
  for pid, cutoff in zip(outcomes['patient_id'], outcomes['prediction_time'])
]
vitals_features_df = pd.DataFrame(vitals_features_rows)
vitals_features_df.head(10)


,patient_id,heart_rate_mean,heart_rate_min,heart_rate_max,heart_rate_std,heart_rate_last,heart_rate_rate_per_hour,temperature_mean,temperature_min,temperature_max,temperature_std,temperature_last,temperature_rate_per_hour,oxygen_saturation_mean,oxygen_saturation_min,oxygen_saturation_max,oxygen_saturation_std,oxygen_saturation_last,oxygen_saturation_rate_per_hour,respiratory_rate_mean,respiratory_rate_min,respiratory_rate_max,respiratory_rate_std,respiratory_rate_last,respiratory_rate_rate_per_hour,blood_pressure_mean,blood_pressure_min,blood_pressure_max,blood_pressure_std,blood_pressure_last,blood_pressure_rate_per_hour
0,1,78.200000,78.2,78.2,0.000000,78.2,0.000000,36.750000,36.75,36.75,0.000000,36.75,0.000000,97.000000,97.0,97.0,0.000000,97.0,0.000000,16.700000,16.7,16.7,0.000000,16.7,0.000000,114.300000,114.3,114.3,0.000000,114.3,0.000000
1,2,77.100000,77.1,77.1,0.000000,77.1,0.000000,36.720000,36.72,36.72,0.000000,36.72,0.000000,97.500000,97.5,97.5,0.000000,97.5,0.000000,13.800000,13.8,13.8,0.000000,13.8,0.000000,99.100000,99.1,99.1,0.000000,99.1,0.000000
2,3,86.900000,86.9,86.9,0.000000,86.9,0.000000,37.540000,37.54,37.54,0.000000,37.54,0.000000,95.900000,95.9,95.9,0.000000,95.9,0.000000,15.300000,15.3,15.3,0.000000,15.3,0.000000,114.800000,114.8,114.8,0.000000,114.8,0.000000
3,4,85.400000,85.4,85.4,0.000000,85.4,0.000000,37.700000,37.70,37.70,0.000000,37.70,0.000000,96.800000,96.8,96.8,0.000000,96.8,0.000000,17.600000,17.6,17.6,0.000000,17.6,0.000000,115.900000,115.9,115.9,0.000000,115.9,0.000000
4,5,78.300000,78.3,78.3,0.000000,78.3,0.000000,36.970000,36.97,36.97,0.000000,36.97,0.000000,98.100000,98.1,98.1,0.000000,98.1,0.000000,13.800000,13.8,13.8,0.000000,13.8,0.000000,118.300000,118.3,118.3,0.000000,118.3,0.000000
5,6,69.900000,65.7,74.1,5.939697,74.1,13.263158,36.285000,36.03,36.54,0.360624,36.54,0.805263,97.450000,96.3,98.6,1.626346,98.6,3.631579,18.250000,17.4,19.1,1.202082,17.4,-2.684211,133.050000,132.5,133.6,0.777817,133.6,1.736842
6,7,72.900000,72.9,72.9,0.000000,72.9,0.000000,37.220000,37.22,37.22,0.000000,37.22,0.000000,95.500000,95.5,95.5,0.000000,95.5,0.000000,10.300000,10.3,10.3,0.000000,10.3,0.000000,110.200000,110.2,110.2,0.000000,110.2,0.000000
7,8,84.366667,78.9,87.1,4.734272,78.9,-3.727273,37.183333,37.17,37.19,0.011547,37.17,-0.009091,95.333333,94.5,96.6,1.115049,96.6,0.954545,15.566667,14.8,16.0,0.665833,16.0,0.045455,124.933333,120.5,127.9,3.911948,126.4,-0.681818
8,9,84.800000,84.8,84.8,0.000000,84.8,0.000000,36.310000,36.31,36.31,0.000000,36.31,0.000000,97.500000,97.5,97.5,0.000000,97.5,0.000000,16.400000,16.4,16.4,0.000000,16.4,0.000000,128.600000,128.6,128.6,0.000000,128.6,0.000000
9,10,71.800000,71.8,71.8,0.000000,71.8,0.000000,37.270000,37.27,37.27,0.000000,37.27,0.000000,98.500000,98.5,98.5,0.000000,98.5,0.000000,17.300000,17.3,17.3,0.000000,17.3,0.000000,133.800000,133.8,133.8,0.000000,133.8,0.000000


In [13]:
# Generate labs features for each patient at their prediction time
LAB_LOOKBACK_HOURS = 24

def labs_features(pid, cutoff, df):
  window=df[
    (df['patient_id'] == pid) & 
    (df['timestamp'] <= cutoff) & 
    (df['timestamp'] >= cutoff - pd.Timedelta(hours=LAB_LOOKBACK_HOURS))
  ]

  if window.empty:
    window = df[(df['patient_id'] == pid) & (df['timestamp'] <= cutoff)].tail(1)
  feats={}
  for col in lab_cols:
    vals = window[col]
    feats[f'{col}_mean'] = vals.mean()
    feats[f'{col}_last'] = vals.iloc[-1] if len(vals) else np.nan
  
  return feats

labs_features_rows = [
  {'patient_id': pid, **labs_features(pid, cutoff, labs)}
  for pid, cutoff in zip(outcomes['patient_id'], outcomes['prediction_time'])
]
labs_features_df = pd.DataFrame(labs_features_rows)
labs_features_df.head(10)

,patient_id,white_cell_count_mean,white_cell_count_last,crp_mean,crp_last,lactate_mean,lactate_last,creatinine_mean,creatinine_last,platelet_count_mean,platelet_count_last
0,1,8.370000,8.49,12.95,20.3,1.565,1.60,1.085000,1.09,220.500000,233.0
1,2,8.110000,8.11,2.70,2.7,0.550,0.55,1.020000,1.02,339.000000,339.0
2,3,7.093333,6.88,6.40,4.9,1.350,1.27,0.886667,0.93,247.333333,235.0
3,4,10.335000,10.94,4.50,5.4,0.925,1.09,1.020000,1.01,316.000000,316.0
4,5,7.420000,7.42,9.40,9.4,1.140,1.14,0.820000,0.82,243.000000,243.0
5,6,10.640000,10.64,6.70,6.7,0.890,0.89,0.670000,0.67,290.000000,290.0
6,7,8.042500,8.56,5.80,6.7,0.320,0.33,0.692500,0.54,329.500000,331.0
7,8,6.905000,7.68,1.85,3.0,0.680,0.68,0.545000,0.65,304.500000,302.0
8,9,7.100000,7.10,5.90,5.9,0.570,0.57,1.300000,1.30,265.000000,265.0
9,10,5.900000,5.90,8.30,8.3,1.010,1.01,1.010000,1.01,291.000000,291.0


In [14]:
patients.columns

Index(['patient_id', 'age', 'gender', 'medical_conditions',
       'registration_date'],
      dtype='str')

In [15]:
# Create a static features dataframe with patient_id, age, gender
static = patients[['patient_id', 'age', 'gender']].copy()

def count_comorbidities(value):
  if pd.isna(value) or value == 'None reported':
    return 0
  return len([item.strip() for item in str(value).split(',') if item.strip()])

static['comorbidity_count'] = patients['medical_conditions'].apply(count_comorbidities)

static = pd.get_dummies(static, columns=['gender'], drop_first=True)
static.head(10)

,patient_id,age,comorbidity_count,gender_Male,gender_Other/Not specified
0,1,66,0,True,False
1,2,42,0,True,False
2,3,74,0,False,False
3,4,77,0,False,False
4,5,25,1,True,False
5,6,37,3,True,False
6,7,63,2,True,False
7,8,55,2,False,False
8,9,60,0,False,True
9,10,45,2,True,False


In [16]:
# Combine static, vitals, labs, and outcomes into a single feature dataframe
feature = (
  static
  .merge(vitals_features_df, on='patient_id')
  .merge(labs_features_df, on='patient_id')
  .merge(outcomes[['patient_id', 'sepsis_event']], on='patient_id')
)

feature_columns = [
  c for c in feature.columns
  if c not in ('patient_id', 'sepsis_event')
]

numeric_cols = feature[feature_columns].select_dtypes(include='number').columns

feature[numeric_cols] = feature[numeric_cols].fillna(feature[numeric_cols].median())

feature['sepsis_event'] = feature['sepsis_event'].astype(int)

print(feature.shape)

(5000, 46)


In [17]:
feature.head(10)

,patient_id,age,comorbidity_count,gender_Male,gender_Other/Not specified,heart_rate_mean,heart_rate_min,heart_rate_max,heart_rate_std,heart_rate_last,heart_rate_rate_per_hour,temperature_mean,temperature_min,temperature_max,temperature_std,temperature_last,temperature_rate_per_hour,oxygen_saturation_mean,oxygen_saturation_min,oxygen_saturation_max,oxygen_saturation_std,oxygen_saturation_last,oxygen_saturation_rate_per_hour,respiratory_rate_mean,respiratory_rate_min,respiratory_rate_max,respiratory_rate_std,respiratory_rate_last,respiratory_rate_rate_per_hour,blood_pressure_mean,blood_pressure_min,blood_pressure_max,blood_pressure_std,blood_pressure_last,blood_pressure_rate_per_hour,white_cell_count_mean,white_cell_count_last,crp_mean,crp_last,lactate_mean,lactate_last,creatinine_mean,creatinine_last,platelet_count_mean,platelet_count_last,sepsis_event
0,1,66,0,True,False,78.200000,78.2,78.2,0.000000,78.2,0.000000,36.750000,36.75,36.75,0.000000,36.75,0.000000,97.000000,97.0,97.0,0.000000,97.0,0.000000,16.700000,16.7,16.7,0.000000,16.7,0.000000,114.300000,114.3,114.3,0.000000,114.3,0.000000,8.370000,8.49,12.95,20.3,1.565,1.60,1.085000,1.09,220.500000,233.0,1
1,2,42,0,True,False,77.100000,77.1,77.1,0.000000,77.1,0.000000,36.720000,36.72,36.72,0.000000,36.72,0.000000,97.500000,97.5,97.5,0.000000,97.5,0.000000,13.800000,13.8,13.8,0.000000,13.8,0.000000,99.100000,99.1,99.1,0.000000,99.1,0.000000,8.110000,8.11,2.70,2.7,0.550,0.55,1.020000,1.02,339.000000,339.0,0
2,3,74,0,False,False,86.900000,86.9,86.9,0.000000,86.9,0.000000,37.540000,37.54,37.54,0.000000,37.54,0.000000,95.900000,95.9,95.9,0.000000,95.9,0.000000,15.300000,15.3,15.3,0.000000,15.3,0.000000,114.800000,114.8,114.8,0.000000,114.8,0.000000,7.093333,6.88,6.40,4.9,1.350,1.27,0.886667,0.93,247.333333,235.0,1
3,4,77,0,False,False,85.400000,85.4,85.4,0.000000,85.4,0.000000,37.700000,37.70,37.70,0.000000,37.70,0.000000,96.800000,96.8,96.8,0.000000,96.8,0.000000,17.600000,17.6,17.6,0.000000,17.6,0.000000,115.900000,115.9,115.9,0.000000,115.9,0.000000,10.335000,10.94,4.50,5.4,0.925,1.09,1.020000,1.01,316.000000,316.0,1
4,5,25,1,True,False,78.300000,78.3,78.3,0.000000,78.3,0.000000,36.970000,36.97,36.97,0.000000,36.97,0.000000,98.100000,98.1,98.1,0.000000,98.1,0.000000,13.800000,13.8,13.8,0.000000,13.8,0.000000,118.300000,118.3,118.3,0.000000,118.3,0.000000,7.420000,7.42,9.40,9.4,1.140,1.14,0.820000,0.82,243.000000,243.0,0
5,6,37,3,True,False,69.900000,65.7,74.1,5.939697,74.1,13.263158,36.285000,36.03,36.54,0.360624,36.54,0.805263,97.450000,96.3,98.6,1.626346,98.6,3.631579,18.250000,17.4,19.1,1.202082,17.4,-2.684211,133.050000,132.5,133.6,0.777817,133.6,1.736842,10.640000,10.64,6.70,6.7,0.890,0.89,0.670000,0.67,290.000000,290.0,0
6,7,63,2,True,False,72.900000,72.9,72.9,0.000000,72.9,0.000000,37.220000,37.22,37.22,0.000000,37.22,0.000000,95.500000,95.5,95.5,0.000000,95.5,0.000000,10.300000,10.3,10.3,0.000000,10.3,0.000000,110.200000,110.2,110.2,0.000000,110.2,0.000000,8.042500,8.56,5.80,6.7,0.320,0.33,0.692500,0.54,329.500000,331.0,1
7,8,55,2,False,False,84.366667,78.9,87.1,4.734272,78.9,-3.727273,37.183333,37.17,37.19,0.011547,37.17,-0.009091,95.333333,94.5,96.6,1.115049,96.6,0.954545,15.566667,14.8,16.0,0.665833,16.0,0.045455,124.933333,120.5,127.9,3.911948,126.4,-0.681818,6.905000,7.68,1.85,3.0,0.680,0.68,0.545000,0.65,304.500000,302.0,0
8,9,60,0,False,True,84.800000,84.8,84.8,0.000000,84.8,0.000000,36.310000,36.31,36.31,0.000000,36.31,0.000000,97.500000,97.5,97.5,0.000000,97.5,0.000000,16.400000,16.4,16.4,0.000000,16.4,0.000000,128.600000,128.6,128.6,0.000000,128.6,0.000000,7.100000,7.10,5.90,5.9,0.570,0.57,1.300000,1.30,265.000000,265.0,0
9,10,45,2,True,False,71.800000,71.8,71.8,0.000000,71.8,0.000000,37.270000,37.27,37.27,0.000000,37.27,0.000000,98.500000,98.5,98.5,0.000000,98.5,0.000000,17.300000,17.3,17.3,0.000000,17.3,0.000000,133.800000,133.8,133.8,0.000000,133.8,0.000000,5.900000,5.90,8.30,8.3,1.010,1.01,1.010000,1.01,291.000000,291.0,1


In [18]:
feature.columns

Index(['patient_id', 'age', 'comorbidity_count', 'gender_Male',
       'gender_Other/Not specified', 'heart_rate_mean', 'heart_rate_min',
       'heart_rate_max', 'heart_rate_std', 'heart_rate_last',
       'heart_rate_rate_per_hour', 'temperature_mean', 'temperature_min',
       'temperature_max', 'temperature_std', 'temperature_last',
       'temperature_rate_per_hour', 'oxygen_saturation_mean',
       'oxygen_saturation_min', 'oxygen_saturation_max',
       'oxygen_saturation_std', 'oxygen_saturation_last',
       'oxygen_saturation_rate_per_hour', 'respiratory_rate_mean',
       'respiratory_rate_min', 'respiratory_rate_max', 'respiratory_rate_std',
       'respiratory_rate_last', 'respiratory_rate_rate_per_hour',
       'blood_pressure_mean', 'blood_pressure_min', 'blood_pressure_max',
       'blood_pressure_std', 'blood_pressure_last',
       'blood_pressure_rate_per_hour', 'white_cell_count_mean',
       'white_cell_count_last', 'crp_mean', 'crp_last', 'lactate_mean',
       'la

In [19]:
# Check mean values of selected features grouped by sepsis event
check_cols = ['heart_rate_last', 'oxygen_saturation_last', 'crp_last', 'lactate_last']
feature.groupby('sepsis_event')[check_cols].mean()

,heart_rate_last,oxygen_saturation_last,crp_last,lactate_last
sepsis_event,,,,
0,77.969741,97.460704,6.318593,1.001574
1,80.875522,97.024391,8.677739,1.057704


In [20]:
# Compute correlation of features with sepsis_event
corr = feature[feature_columns + ['sepsis_event']].corr()['sepsis_event'].drop('sepsis_event')
corr.sort_values(key=abs, ascending=False).head(10)

age                       0.669143
comorbidity_count         0.250626
crp_last                  0.192154
oxygen_saturation_last   -0.174935
heart_rate_last           0.153710
crp_mean                  0.150791
oxygen_saturation_min    -0.143097
respiratory_rate_last     0.141411
oxygen_saturation_mean   -0.122410
heart_rate_max            0.120342
Name: sepsis_event, dtype: float64

In [21]:
# Save the final feature dataframe to a CSV file
feature.to_csv('../data/processed/sepsis_features.csv', index=False)